# NB9 — Token-length diagnostic: choose K (chunks/article) from the data, not a guess

**Why.** The `1235 > 512` line in NB8 was NOT the longest article — it was just the FIRST article the
tokenizer flagged (it warns once or twice then goes quiet). Articles reach ~3500 words ≈ 5000–7000
tokens ≈ 10–14 chunks. So K=4 would truncate the long tail badly. This notebook MEASURES the real
token-length distribution for BOTH encoders and prints, for each K, the % of articles fully covered
(no truncation) with chunk size 510 + stride 460 — the exact chunking used in extraction/fine-tuning.

CPU only, seconds. No embeddings, no GPU. Reads only the dataset text.

## 1 · Config

In [1]:
import numpy as np, pandas as pd
P_DATASET = "/kaggle/input/notebooks/bahaaqassem/nb3-build-dataset/dataset.parquet"   # EDIT
MAX_CT, STRIDE = 510, 460            # content tokens per chunk, sliding stride (must match the pipeline)
ENCODERS = {"CAMeLBERT-MSA": "CAMeL-Lab/bert-base-arabic-camelbert-msa",
            "XLM-R":         "xlm-roberta-base"}
print("config loaded")

config loaded


## 2 · Load text

In [2]:
df = pd.read_parquet(P_DATASET)
if "article_id" in df.columns: df = df.set_index("article_id")
texts = df["text"].astype(str).tolist()
lab   = df["label"].to_numpy()
print(f"{len(texts)} articles | words: median {int(np.median([len(t.split()) for t in texts]))}, "
      f"max {max(len(t.split()) for t in texts)}")

7101 articles | words: median 589, max 5524


## 3 · Tokenize once per encoder, count tokens + chunks

In [3]:
!pip install -q transformers
from transformers import AutoTokenizer

def n_chunks(n_tok):
    # how many sliding windows of size MAX_CT (stride STRIDE) cover n_tok content tokens
    if n_tok <= MAX_CT: return 1
    return 1 + int(np.ceil((n_tok - MAX_CT) / STRIDE))

per_enc = {}
for name, mid in ENCODERS.items():
    tok = AutoTokenizer.from_pretrained(mid)
    ntok = np.array([len(tok(t, add_special_tokens=False)["input_ids"]) for t in texts])
    nch  = np.array([n_chunks(x) for x in ntok])
    per_enc[name] = {"ntok": ntok, "nch": nch}
    print(f"{name}: tokens  median {int(np.median(ntok))}  p90 {int(np.percentile(ntok,90))}  "
          f"p95 {int(np.percentile(ntok,95))}  p99 {int(np.percentile(ntok,99))}  max {int(ntok.max())}")

config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

CAMeLBERT-MSA: tokens  median 793  p90 1611  p95 2138  p99 4056  max 17755


...

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1235 > 512). Running this sequence through the model will result in indexing errors


XLM-R: tokens  median 1011  p90 2070  p95 2713  p99 5061  max 9984


## 4 · Token-length percentiles + chunk distribution

In [4]:
import IPython.display as ipd
rows=[]
for name,d in per_enc.items():
    t=d["ntok"]
    rows.append({"encoder":name, "median":int(np.median(t)), "p90":int(np.percentile(t,90)),
                 "p95":int(np.percentile(t,95)), "p99":int(np.percentile(t,99)), "max":int(t.max()),
                 "chunks_median":int(np.median(d["nch"])), "chunks_max":int(d["nch"].max())})
print("TOKEN LENGTH (content tokens):"); ipd.display(pd.DataFrame(rows))

TOKEN LENGTH (content tokens):


,encoder,median,p90,p95,p99,max,chunks_median,chunks_max
0,CAMeLBERT-MSA,793,1611,2138,4056,17755,2,39
1,XLM-R,1011,2070,2713,5061,9984,3,22


## 5 · Coverage table — for each K, % of articles FULLY covered (no truncation)

In [5]:
Ks = [1,2,3,4,5,6,8,10,12,16]
cov=[]
for name,d in per_enc.items():
    nch=d["nch"]; row={"encoder":name}
    for K in Ks:
        row[f"K={K}"] = round(100*np.mean(nch<=K), 1)
    cov.append(row)
covdf=pd.DataFrame(cov)
print("% of articles fully covered by first-K chunks (chunk=510, stride=460):")
ipd.display(covdf)

# smallest K reaching 95% and 99% coverage, per encoder
print("\nSmallest K for target coverage:")
for name,d in per_enc.items():
    nch=d["nch"]
    k95=next((K for K in range(1, nch.max()+1) if np.mean(nch<=K)>=0.95), nch.max())
    k99=next((K for K in range(1, nch.max()+1) if np.mean(nch<=K)>=0.99), nch.max())
    print(f"  {name}:  K>=95% = {k95}   K>=99% = {k99}   (full = {nch.max()})")

% of articles fully covered by first-K chunks (chunk=510, stride=460):


,encoder,K=1,K=2,K=3,K=4,K=5,K=6,K=8,K=10,K=12,K=16
0,CAMeLBERT-MSA,1.9,67.6,86.7,93.1,96.4,97.5,98.5,99.4,99.8,99.9
1,XLM-R,0.0,45.7,75.7,87.7,92.8,95.5,97.6,98.6,99.3,99.9



Smallest K for target coverage:
  CAMeLBERT-MSA:  K>=95% = 5   K>=99% = 9   (full = 39)
  XLM-R:  K>=95% = 6   K>=99% = 11   (full = 22)


## 6 · What truncation at a given K costs (tokens dropped)

In [6]:
K_TEST = 4   # change to inspect any cap
for name,d in per_enc.items():
    ntok=d["ntok"]; cap = MAX_CT + (K_TEST-1)*STRIDE   # tokens kept by first-K chunks
    dropped = np.clip(ntok-cap, 0, None)
    aff = np.mean(ntok>cap)*100
    print(f"{name} @ K={K_TEST} (keeps ~{cap} tok): {aff:.1f}% of articles truncated; "
          f"mean tokens dropped among those = {dropped[dropped>0].mean():.0f}" if (dropped>0).any()
          else f"{name} @ K={K_TEST}: no truncation")
print("\nRule: pick the smallest K that covers ~95-99% fully; if that K is too big for GPU memory,")
print("that's the signal to use hierarchical pooling / partial layer-freezing instead of plain first-K.")

CAMeLBERT-MSA @ K=4 (keeps ~1890 tok): 6.9% of articles truncated; mean tokens dropped among those = 1070
XLM-R @ K=4 (keeps ~1890 tok): 12.3% of articles truncated; mean tokens dropped among those = 1086

Rule: pick the smallest K that covers ~95-99% fully; if that K is too big for GPU memory,
that's the signal to use hierarchical pooling / partial layer-freezing instead of plain first-K.
